In [1]:
# Path + packaging
import sys  # no installation needed
from pathlib import Path  # no installation needed

SRC_DIR = Path(r"C:\Users\quantbase\Desktop\SyStrat\src")
PKG_DIR = SRC_DIR / "syslib"
PKG_DIR.mkdir(parents=True, exist_ok=True)
(PKG_DIR / "__init__.py").touch(exist_ok=True)  # ensure it's importable

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("sys.path OK:", SRC_DIR in map(Path, map(str, sys.path)))


sys.path OK: True


In [2]:
import importlib  # no installation needed
import pandas as pd
from datetime import date  # no installation needed
import json  # no installation needed

SRC_DIR = Path(r"C:\Users\quantbase\Desktop\SyStrat\src")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import syslib.wp_core as wp_core   
import syslib.wp_ml_prep as wp_ml_prep 
import syslib.wp_ml_features as wp_ml_feat 
import syslib.wp_bt as wp_bt
importlib.reload(wp_core)
importlib.reload(wp_ml_prep)     # labels (done already)
importlib.reload(wp_ml_prep)
importlib.reload(wp_ml_feat)
importlib.reload(wp_bt)

from syslib.wp_ml_train import run_step4
from syslib.wp_ml_train import run_step4_with_report  # no installation needed

In [3]:
import numpy as np
from xgboost import XGBRegressor

In [4]:
RUN_DATE = date.today().strftime("%d-%m-%Y") #"24-10-2025"   
BASE = Path(r"C:\Users\quantbase\Desktop\SyStrat") / RUN_DATE
ml_dir = BASE / "data_int" / "ml"
BASE

WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025')

In [5]:
# Build (prefers log_returns.csv; falls back to close.csv)
labels = wp_ml_prep.build_labels_from_files(
    base_dir=BASE,
    ks=(5, 10, 30),
)
list(labels.keys())  # [5, 10, 20]

[5, 10, 30]

In [6]:
paths = wp_ml_prep.save_labels(labels, base_dir=BASE, prefix="labels_k")
paths

{'k5_csv': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/labels_k5.csv'),
 'k5_parquet': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/labels_k5.parquet'),
 'k10_csv': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/labels_k10.csv'),
 'k10_parquet': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/labels_k10.parquet'),
 'k30_csv': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/labels_k30.csv'),
 'k30_parquet': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/labels_k30.parquet')}

In [7]:
#-------Build features, impute (time-safe), split, scale, and save-----

In [8]:
paths = wp_ml_feat.run_step_3_2(
    base_dir=BASE,
    k_label=10,
    do_scale=True,
    split_frac=(0.40, 0.35, 0.25)
)
paths

{'X_train': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/X_train_k10.parquet'),
 'y_train': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/y_train_k10.parquet'),
 'X_val': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/X_val_k10.parquet'),
 'y_val': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/y_val_k10.parquet'),
 'X_test': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/X_test_k10.parquet'),
 'y_test': WindowsPath('C:/Users/quantbase/Desktop/SyStrat/17-12-2025/data_int/ml/y_test_k10.parquet')}

In [9]:
Xtr = pd.read_parquet(ml_dir / "X_train_k10.parquet")
ytr = pd.read_parquet(ml_dir / "y_train_k10.parquet")

In [10]:
Xtr.to_csv((ml_dir / "X_train_k10.csv"))
ytr.to_csv((ml_dir / "y_train_k10.csv"))

In [11]:
corr = pd.concat([Xtr, ytr.rename(columns={"y":"label"})], axis=1).corr(numeric_only=True)
corr["label"].sort_values(ascending=False).head(15)

label      1.000000
ewma94     0.427416
rv10       0.411683
rv5        0.398043
park_d     0.388497
rv20       0.387601
vov        0.234268
mon_5      0.151853
slope20    0.119358
d_rv10     0.090181
mon_1      0.078768
mon_2      0.062756
mon_3      0.043344
dow_1      0.022390
mon_8      0.020369
Name: label, dtype: float64

In [12]:
Xtr

,park_d,ema9,macd21,slope20,rv5,rv10,rv20,ewma94,d_rv10,vov,...,mon_3,mon_4,mon_5,mon_6,mon_7,mon_8,mon_9,mon_10,mon_11,mon_12
0,-0.477135,-0.376247,-0.057071,0.972348,-0.224335,-0.523331,-0.455301,-0.567282,0.075820,-0.253802,...,-0.342802,-0.33659,-0.344961,-0.307717,-0.292608,-0.285373,-0.276945,-0.281883,-0.276945,-0.281883
1,0.657006,-0.376247,-0.057071,0.802330,0.273740,-0.268496,-0.283797,-0.360259,0.875834,-0.337111,...,-0.342802,-0.33659,-0.344961,-0.307717,-0.292608,-0.285373,-0.276945,-0.281883,-0.276945,-0.281883
2,1.091414,-0.376247,-0.057071,0.652466,0.193715,-0.258820,-0.372576,-0.407076,0.034071,-0.310449,...,-0.342802,-0.33659,-0.344961,-0.307717,-0.292608,-0.285373,-0.276945,-0.281883,-0.276945,-0.281883
3,0.424962,-0.376247,-0.057071,0.536120,-0.023980,-0.199194,-0.528842,-0.416242,0.205575,-0.214599,...,-0.342802,-0.33659,-0.344961,-0.307717,-0.292608,-0.285373,-0.276945,-0.281883,-0.276945,-0.281883
4,-0.058027,-0.376247,-0.057071,0.456994,0.067117,-0.144478,-0.495845,-0.426800,0.188713,-0.141791,...,-0.342802,-0.33659,-0.344961,-0.307717,-0.292608,-0.285373,-0.276945,-0.281883,-0.276945,-0.281883
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5891,-0.736573,-0.376221,-0.057163,-0.281831,-0.871862,-0.646585,-0.764390,-0.463384,-0.077561,-0.521021,...,-0.342802,-0.33659,-0.344961,3.249736,-0.292608,-0.285373,-0.276945,-0.281883,-0.276945,-0.281883
5892,-0.086491,-0.376221,-0.057160,-0.265027,-0.898069,-0.646844,-0.912204,-0.512534,-0.000042,-0.591640,...,-0.342802,-0.33659,-0.344961,3.249736,-0.292608,-0.285373,-0.276945,-0.281883,-0.276945,-0.281883
5893,-0.733184,-0.376221,-0.057157,-0.231258,-1.072306,-0.635015,-0.936289,-0.555523,0.041460,-0.823639,...,-0.342802,-0.33659,-0.344961,3.249736,-0.292608,-0.285373,-0.276945,-0.281883,-0.276945,-0.281883
5894,-1.025121,-0.376221,-0.057154,-0.214682,-1.075433,-0.911950,-0.949931,-0.605336,-0.950025,-0.661061,...,-0.342802,-0.33659,-0.344961,3.249736,-0.292608,-0.285373,-0.276945,-0.281883,-0.276945,-0.281883


In [13]:
ytr

,y
0,0.050488
1,0.041853
2,0.058534
3,0.056983
4,0.055863
...,...
5891,0.058263
5892,0.058889
5893,0.060132
5894,0.062744


In [14]:
Xtr.isna().sum().sum()

0

In [15]:
Xtr.shape

(5896, 32)

In [16]:
len(ytr)

5896

In [17]:
#-----Model train/validate/test-----

In [18]:
#metrics = run_step4(BASE, k_label=10, n_splits=4, use_alpha_calibrator=True)
#metrics

In [19]:
REPORT_DIR =  Path(r"C:\Users\quantbase\Desktop\SyStrat") / RUN_DATE # ROOT of the run (not ...\data_int\ml)

In [20]:
out = run_step4_with_report(REPORT_DIR, k_label=10, n_splits=4, use_alpha_calibrator=True, model_for_report='xgb')
print(out["report"])

{'k_label': 10, 'paths': {'deciles_val': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\17-12-2025\\data_int\\ml\\reports_k10\\deciles_val.csv', 'deciles_test': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\17-12-2025\\data_int\\ml\\reports_k10\\deciles_test.csv', 'calibration_val_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\17-12-2025\\data_int\\ml\\reports_k10\\calibration_val.png', 'calibration_test_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\17-12-2025\\data_int\\ml\\reports_k10\\calibration_test.png', 'residuals_val_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\17-12-2025\\data_int\\ml\\reports_k10\\residuals_val.png', 'residuals_test_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\17-12-2025\\data_int\\ml\\reports_k10\\residuals_test.png', 'by_ticker_val_csv': None, 'by_ticker_test_csv': None, 'feature_importance_xgb_csv': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\17-12-2025\\data_int\\ml\\reports_k10\\feature_importance_xgb.csv', 'feature_importance_xgb_png': 'C:\\Users\\quantbase\\Des

In [21]:
pred_test = pd.read_parquet(ml_dir / "preds_test_k10.parquet")
pred_val = pd.read_parquet(ml_dir / "preds_val_k10.parquet")

In [22]:
pred_test.to_csv(ml_dir / "preds_test_k10.csv")
pred_val.to_csv(ml_dir / "preds_val_k10.csv")

In [23]:
# Report using ElasticNet predictions
out_en = run_step4_with_report(REPORT_DIR, k_label=10, n_splits=4, use_alpha_calibrator=True, model_for_report='en')
print(out_en["report"])

FileNotFoundError: Neither X_train_k10.parquet nor X_train_k10.csv found at C:\Users\quantbase\Desktop\SyStrat\24-10-2025\data_int\ml\data_int\ml

In [10]:
out_en = run_step4_with_report(REPORT_DIR, k_label=14, n_splits=4, use_alpha_calibrator=True, model_for_report='gbm')
print(out_en["report"])

{'k_label': 10, 'paths': {'deciles_val': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k10\\deciles_val.csv', 'deciles_test': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k10\\deciles_test.csv', 'calibration_val_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k10\\calibration_val.png', 'calibration_test_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k10\\calibration_test.png', 'residuals_val_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k10\\residuals_val.png', 'residuals_test_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k10\\residuals_test.png', 'by_ticker_val_csv': None, 'by_ticker_test_csv': None, 'feature_importance_csv': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k10\\feature_importance_gbm.csv', 'feature_importance_png': 'C:\\Users\\quantbase\\Desktop\\Sy

In [ ]:
#out_en = run_step4_with_report(Path(r"C:\Users\quantbase\Desktop\SyStrat\14-10-2025\data_int\ml\reports_k10") / , k_label=10, n_splits=4, use_alpha_calibrator=True, model_for_report='en')

In [23]:
# A) One-shot: train + report (plots/deciles use GBM; importances also GBM by default)
out = run_step4_with_report(REPORT_DIR, k_label=20, n_splits=4, use_alpha_calibrator=True, model_for_report='gbm')
print(out["report"]["paths"])


{'deciles_val': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\deciles_val.csv', 'deciles_test': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\deciles_test.csv', 'calibration_val_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\calibration_val.png', 'calibration_test_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\calibration_test.png', 'residuals_val_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\residuals_val.png', 'residuals_test_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\residuals_test.png', 'by_ticker_val_csv': None, 'by_ticker_test_csv': None, 'feature_importance_gbm_csv': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\feature_importance_gbm.csv', 'feature_importance_gbm_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025

In [24]:
# D) XGB-centered report (plots use XGB; importances default to XGB)
out = run_step4_with_report(BASE, k_label=20, n_splits=4, use_alpha_calibrator=True, model_for_report='xgb')
print(out["report"]["paths"])

{'deciles_val': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\deciles_val.csv', 'deciles_test': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\deciles_test.csv', 'calibration_val_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\calibration_val.png', 'calibration_test_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\calibration_test.png', 'residuals_val_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\residuals_val.png', 'residuals_test_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\residuals_test.png', 'by_ticker_val_csv': None, 'by_ticker_test_csv': None, 'feature_importance_xgb_csv': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025\\data_int\\ml\\reports_k20\\feature_importance_xgb.csv', 'feature_importance_xgb_png': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\24-10-2025

In [28]:
X = np.random.randn(200, 15).astype('float32')
y = np.abs(np.random.randn(200).astype('float32'))
XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=3,
    subsample=0.7, colsample_bytree=0.7, reg_alpha=0.0, reg_lambda=1.0,
    tree_method="hist", n_jobs=4, random_state=42
).fit(X, y)
print("xgboost OK")


xgboost OK


In [60]:
#--------------legend-------------